# LeetCode 3005 — Count Elements With Maximum Frequency bằng Python

Link đề: https://leetcode.com/problems/count-elements-with-maximum-frequency/ · Độ khó: **Easy** · Chủ đề: Array, Hash Table, Counting

Skeleton LeetCode:

```python
from typing import List

class Solution:
    def maxFrequencyElements(self, nums: List[int]) -> int:
        ...
```

Ghi chú: đề và các ví dụ được viết theo trí nhớ về đề gốc; mọi đáp án trong notebook đã được kiểm chứng bằng code, nhưng nên đối chiếu lại với đề gốc nếu muốn khớp từng chữ.

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: **tần suất** của một số là số lần nó xuất hiện; trả về **tổng tần suất** của **mọi** số có tần suất **lớn nhất** (không phải số lượng số, cũng không phải bản thân tần suất lớn nhất).
2. Biết **đếm tần suất** bằng `dict` thủ công, `collections.Counter` và **mảng đếm** (vì giá trị nằm trong `1..100`).
3. Hiểu quan hệ `đáp án = (tần suất lớn nhất) × (số lượng giá trị đạt tần suất đó)`.
4. Đi từ cách **đếm bằng `Counter`, rồi `max` và cộng** → **`dict` thủ công** → **mảng đếm theo giá trị** → **một lượt duyệt duy trì `max` và `total` đang chạy**.
5. Tránh các lỗi kinh điển: trả về tần suất lớn nhất, trả về số lượng giá trị (thay vì tổng số lần xuất hiện), `max` trên **khóa** thay vì **giá trị** của dict, và đặt lại `total` sai khi xuất hiện kỷ lục mới.

## 2. Đề bài & ví dụ

Cho mảng số nguyên dương `nums`. Hãy trả về **tổng tần suất** của các phần tử trong `nums` sao cho các phần tử đó có **tần suất lớn nhất**. **Tần suất** của một phần tử là **số lần nó xuất hiện** trong mảng.

**Ví dụ 1**

- Input: `nums = [1, 2, 2, 3, 1, 4]`
- Output: `4`
- Giải thích: các phần tử `1` và `2` đều xuất hiện `2` lần, đó là tần suất lớn nhất. Số phần tử có tần suất lớn nhất là `2 + 2 = 4`, nên đáp án là `4`.

**Ví dụ 2**

- Input: `nums = [1, 2, 3, 4, 5]`
- Output: `5`
- Giải thích: mọi phần tử đều xuất hiện `1` lần (tần suất lớn nhất là `1`). Có `5` phần tử như vậy, tổng tần suất `5`.

**Ràng buộc:** `1 <= nums.length <= 100`; `1 <= nums[i] <= 100`.

In [1]:
from typing import List
from collections import Counter
import random

examples = [
    ([1, 2, 2, 3, 1, 4], 4),
    ([1, 2, 3, 4, 5], 5),
    ([7], 1),
    ([5, 5, 5, 2, 2], 3),
]
for nums, expected in examples:
    print("nums =", nums, "-> kỳ vọng", expected)

nums = [1, 2, 2, 3, 1, 4] -> kỳ vọng 4
nums = [1, 2, 3, 4, 5] -> kỳ vọng 5
nums = [7] -> kỳ vọng 1
nums = [5, 5, 5, 2, 2] -> kỳ vọng 3


## 3. Kiến thức nền

### 3.1. Tần suất (frequency) và `list.count`

`a.count(x)` trả về số lần `x` xuất hiện trong list `a` (quét cả list, `O(n)`). Tần suất của mỗi giá trị khác nhau là thứ ta cần so sánh.

In [2]:
nums = [1, 2, 2, 3, 1, 4]
print(nums.count(1), nums.count(2), nums.count(3), nums.count(4), nums.count(9))   # 2 2 1 1 0

2 2 1 1 0


### 3.2. Đếm bằng `dict`

Dùng `dict` ánh xạ `giá_trị -> số_lần`. Mỗi lần gặp `x`: `freq[x] = freq.get(x, 0) + 1`. `get(x, 0)` trả `0` nếu `x` chưa có khóa, tránh lỗi `KeyError`.

In [3]:
nums = [1, 2, 2, 3, 1, 4]
freq = {}
for x in nums:
    freq[x] = freq.get(x, 0) + 1
print(freq)                                # {1: 2, 2: 2, 3: 1, 4: 1}
print(freq.keys(), freq.values())

{1: 2, 2: 2, 3: 1, 4: 1}
dict_keys([1, 2, 3, 4]) dict_values([2, 2, 1, 1])


### 3.3. `collections.Counter`

`Counter(nums)` làm đúng việc trên chỉ với một lệnh và trả về một `dict` đặc biệt: truy cập khóa không có cho `0`. Có `.values()` (các tần suất), `.most_common()` (sắp xếp giảm dần theo tần suất).

In [4]:
c = Counter([1, 2, 2, 3, 1, 4])
print(c)                                   # Counter({1: 2, 2: 2, 3: 1, 4: 1})
print(c[9])                                # 0: không báo lỗi
print(list(c.values()), c.most_common(2))

Counter({1: 2, 2: 2, 3: 1, 4: 1})
0
[2, 2, 1, 1] [(1, 2), (2, 2)]


### 3.4. `max` trên giá trị của dict (không phải khóa!)

`max(freq)` trả về **khóa lớn nhất** (số lớn nhất xuất hiện trong mảng) chứ **không** phải tần suất lớn nhất. Muốn tần suất lớn nhất phải viết `max(freq.values())`. Hai thứ này rất dễ nhầm.

In [5]:
freq = {1: 2, 2: 2, 3: 1, 9: 1}
print(max(freq))                           # 9  <- khóa lớn nhất
print(max(freq.values()))                  # 2  <- tần suất lớn nhất

9
2


### 3.5. Đếm có điều kiện với `sum`

`sum(1 for v in freq.values() if v == mx)` đếm số giá trị có tần suất bằng `mx`. `sum(v for v in freq.values() if v == mx)` cộng chính các tần suất đó (chính là đáp án).

In [6]:
freq = {1: 2, 2: 2, 3: 1, 4: 1}
mx = max(freq.values())
print(sum(1 for v in freq.values() if v == mx))      # 2: số giá trị đạt tần suất lớn nhất
print(sum(v for v in freq.values() if v == mx))      # 4: tổng tần suất = đáp án
print(mx * sum(1 for v in freq.values() if v == mx)) # 4: cùng kết quả (mx × số giá trị)

2
4
4


### 3.6. Mảng đếm theo giá trị

Vì `1 <= nums[i] <= 100`, ta dùng `cnt = [0] * 101` rồi `cnt[x] += 1`. Không cần hàm băm, truy cập `O(1)` thật sự; chỉ số `0` không dùng.

In [7]:
nums = [1, 2, 2, 3, 1, 4]
cnt = [0] * 101
for x in nums:
    cnt[x] += 1
print(cnt[:6])                             # [0, 2, 2, 1, 1, 0]
print(max(cnt))                            # 2

[0, 2, 2, 1, 1, 0]
2


## 4. Ý tưởng nền tảng / cách nghĩ trực giác (tính tay)

Với `nums = [1, 2, 2, 3, 1, 4]` làm hai bước: **đếm** rồi **tổng hợp**.

| giá trị | tần suất |
|---|---|
| 1 | 2 |
| 2 | 2 |
| 3 | 1 |
| 4 | 1 |

- Tần suất lớn nhất `mx = 2`.
- Các giá trị đạt `mx`: `1` và `2` (hai giá trị).
- Đáp án = cộng tần suất của chúng `= 2 + 2 = 4` (hay `mx × 2 = 4`).

Ví dụ 2 `[1, 2, 3, 4, 5]`: mỗi giá trị tần suất `1`, `mx = 1`, có `5` giá trị đạt `mx`, đáp án `1 × 5 = 5`.

In [8]:
nums = [1, 2, 2, 3, 1, 4]
freq = Counter(nums)
mx = max(freq.values())
winners = [v for v, f in freq.items() if f == mx]
print("tần suất:", dict(freq))
print("mx =", mx, "| các giá trị đạt mx:", winners, "| đáp án =", mx * len(winners))

tần suất: {1: 2, 2: 2, 3: 1, 4: 1}
mx = 2 | các giá trị đạt mx: [1, 2] | đáp án = 4


## 5. Cách 1 — Brute Force / baseline: `Counter`, rồi `max` và cộng

**Ý tưởng.** Đếm bằng `Counter`. Tìm `mx = max(freq.values())`. Cộng mọi tần suất bằng `mx`: `sum(f for f in freq.values() if f == mx)`.

In [9]:
def max_frequency_elements_counter(nums: List[int]) -> int:
    freq = Counter(nums)                               # giá trị -> số lần
    mx = max(freq.values())                            # tần suất lớn nhất
    return sum(f for f in freq.values() if f == mx)    # tổng tần suất của các giá trị đạt mx


for nums, expected in examples:
    print(nums, "->", max_frequency_elements_counter(nums), "(kỳ vọng", expected, ")")

[1, 2, 2, 3, 1, 4] -> 4 (kỳ vọng 4 )
[1, 2, 3, 4, 5] -> 5 (kỳ vọng 5 )
[7] -> 1 (kỳ vọng 1 )
[5, 5, 5, 2, 2] -> 3 (kỳ vọng 3 )


In [10]:
# Xem từng bước
nums = [1, 2, 2, 3, 1, 4]
freq = Counter(nums)
print("freq:", dict(freq))
mx = max(freq.values())
print("mx:", mx)
for v, f in freq.items():
    print(f"giá trị {v}: tần suất {f} {'== mx -> cộng' if f == mx else '< mx -> bỏ qua'}")

freq: {1: 2, 2: 2, 3: 1, 4: 1}
mx: 2
giá trị 1: tần suất 2 == mx -> cộng
giá trị 2: tần suất 2 == mx -> cộng
giá trị 3: tần suất 1 < mx -> bỏ qua
giá trị 4: tần suất 1 < mx -> bỏ qua


**Độ phức tạp (Cách 1).** Đếm `O(n)`, `max` và `sum` trên các giá trị khác nhau (tối đa `n`): **Time `O(n)`**; `Counter` có tối đa `n` khóa: **Space `O(n)`**.

**Nút thắt.** Đã tuyến tính, tối ưu về bậc. Ta chỉ có thể giảm số lượt duyệt (một lượt thay vì ba) hoặc bỏ `Counter`.

## 6. Ý tưởng thuật toán / bước đột phá

Không có bước đột phá về bậc; có hai cải tiến về **cách viết** và **số lượt duyệt**:

1. **Tự đếm** bằng `dict` hoặc mảng đếm (vì giá trị `<= 100`): ít phụ thuộc thư viện, giảng rõ cơ chế.
2. **Duy trì `max` và `total` khi đếm** trong **một lượt**: mỗi lần tăng tần suất của `x` lên `c`:
   - nếu `c > maxf`: kỷ lục mới, `maxf = c`, `total = c` (chỉ `x` đang đạt kỷ lục, các giá trị khác thấp hơn);
   - nếu `c == maxf`: `x` vừa **bằng** kỷ lục, `total += c` (thêm toàn bộ `c` lần xuất hiện của `x`);
   - nếu `c < maxf`: không ảnh hưởng.
3. Ngoài ra: đáp án luôn bằng **`mx × (số giá trị đạt mx)`**.

## 7. Các cách tiếp theo

### Cách 2 — `dict` thủ công

**Giải thích.** Không dùng `Counter`: tự duyệt, `freq[x] = freq.get(x, 0) + 1`; rồi hai vòng: tìm `mx`, cộng các tần suất bằng `mx`.

In [11]:
def max_frequency_elements_dict(nums: List[int]) -> int:
    freq = {}
    for x in nums:
        freq[x] = freq.get(x, 0) + 1                   # đếm thủ công
    mx = 0
    for f in freq.values():
        if f > mx:
            mx = f
    total = 0
    for f in freq.values():
        if f == mx:
            total += f
    return total


for nums, expected in examples:
    print(nums, "->", max_frequency_elements_dict(nums), "(kỳ vọng", expected, ")")

[1, 2, 2, 3, 1, 4] -> 4 (kỳ vọng 4 )
[1, 2, 3, 4, 5] -> 5 (kỳ vọng 5 )
[7] -> 1 (kỳ vọng 1 )
[5, 5, 5, 2, 2] -> 3 (kỳ vọng 3 )


**Độ phức tạp Cách 2:** **Time `O(n)`**, **Space `O(n)`**.

### Cách 3 — Mảng đếm theo giá trị

**Giải thích.** `cnt = [0] * 101`; đếm; `mx = max(cnt)`; đáp án `= mx * cnt.count(mx)`. (`cnt.count(mx)` đếm có bao nhiêu giá trị đạt `mx`; chỉ số `0` luôn có `cnt[0] = 0 < mx` vì `mx >= 1`, nên không gây nhầm.)

In [12]:
def max_frequency_elements_array(nums: List[int]) -> int:
    cnt = [0] * 101                                    # cnt[v] = số lần v xuất hiện
    for x in nums:
        cnt[x] += 1
    mx = max(cnt)
    return mx * cnt.count(mx)                          # mx × số giá trị đạt mx


for nums, expected in examples:
    print(nums, "->", max_frequency_elements_array(nums), "(kỳ vọng", expected, ")")

[1, 2, 2, 3, 1, 4] -> 4 (kỳ vọng 4 )
[1, 2, 3, 4, 5] -> 5 (kỳ vọng 5 )
[7] -> 1 (kỳ vọng 1 )
[5, 5, 5, 2, 2] -> 3 (kỳ vọng 3 )


**Độ phức tạp Cách 3:** **Time `O(n + V)`** với `V = 101`, **Space `O(V)`** (hằng số).

### Cách 4 — Một lượt, duy trì `maxf` và `total` đang chạy

**Giải thích.** Vừa đếm vừa cập nhật kỷ lục như ở mục 6. Không cần lượt thứ hai để tìm `max` và tổng.

In [13]:
def max_frequency_elements_one_pass(nums: List[int]) -> int:
    cnt = [0] * 101
    maxf = 0                                           # tần suất lớn nhất đến giờ
    total = 0                                          # tổng tần suất của các giá trị đạt maxf
    for x in nums:
        cnt[x] += 1
        c = cnt[x]
        if c > maxf:
            maxf = c
            total = c                                  # kỷ lục mới: chỉ x đạt kỷ lục
        elif c == maxf:
            total += c                                 # x vừa bằng kỷ lục: thêm cả c lần xuất hiện của x
    return total


for nums, expected in examples:
    print(nums, "->", max_frequency_elements_one_pass(nums), "(kỳ vọng", expected, ")")

# Quan sát maxf và total chạy qua [1, 2, 2, 3, 1, 4]
cnt, maxf, total = [0] * 101, 0, 0
for x in [1, 2, 2, 3, 1, 4]:
    cnt[x] += 1
    c = cnt[x]
    if c > maxf:
        maxf, total = c, c
    elif c == maxf:
        total += c
    print(f"x={x}: cnt[x]={c} | maxf={maxf} total={total}")

[1, 2, 2, 3, 1, 4] -> 4 (kỳ vọng 4 )
[1, 2, 3, 4, 5] -> 5 (kỳ vọng 5 )
[7] -> 1 (kỳ vọng 1 )
[5, 5, 5, 2, 2] -> 3 (kỳ vọng 3 )
x=1: cnt[x]=1 | maxf=1 total=1
x=2: cnt[x]=1 | maxf=1 total=2
x=2: cnt[x]=2 | maxf=2 total=2
x=3: cnt[x]=1 | maxf=2 total=2
x=1: cnt[x]=2 | maxf=2 total=4
x=4: cnt[x]=1 | maxf=2 total=4


**Độ phức tạp Cách 4:** **Time `O(n)`** (một lượt), **Space `O(V)`**.

In [14]:
# Code nộp LeetCode cho Cách 1 (rõ ràng, ngắn)
from typing import List
from collections import Counter

class Solution:
    def maxFrequencyElements(self, nums: List[int]) -> int:
        freq = Counter(nums)
        mx = max(freq.values())
        return sum(f for f in freq.values() if f == mx)

## 8. TRACE từng bước

Ta theo dõi (a) Cách 4 (một lượt) trên `[1, 2, 2, 3, 1, 4, 4]` và (b) Cách 1 với bảng tần suất.

In [15]:
def trace_one_pass(nums):
    cnt = [0] * 101
    maxf, total = 0, 0
    print("nums =", nums)
    for i, x in enumerate(nums):
        cnt[x] += 1
        c = cnt[x]
        if c > maxf:
            action = f"c={c} > maxf={maxf}: kỷ lục mới -> total = {c}"
            maxf, total = c, c
        elif c == maxf:
            action = f"c={c} == maxf: bằng kỷ lục -> total += {c}"
            total += c
        else:
            action = f"c={c} < maxf={maxf}: bỏ qua"
        print(f"i={i} x={x}: {action:<46} | maxf={maxf} total={total}")
    print("=> kết quả:", total)
    return total


assert trace_one_pass([1, 2, 2, 3, 1, 4, 4]) == 6          # 1, 2, 4 đều xuất hiện 2 lần: 6
assert trace_one_pass([1, 2, 2, 3, 1, 4]) == 4

nums = [1, 2, 2, 3, 1, 4, 4]
i=0 x=1: c=1 > maxf=0: kỷ lục mới -> total = 1          | maxf=1 total=1
i=1 x=2: c=1 == maxf: bằng kỷ lục -> total += 1         | maxf=1 total=2
i=2 x=2: c=2 > maxf=1: kỷ lục mới -> total = 2          | maxf=2 total=2
i=3 x=3: c=1 < maxf=2: bỏ qua                           | maxf=2 total=2
i=4 x=1: c=2 == maxf: bằng kỷ lục -> total += 2         | maxf=2 total=4
i=5 x=4: c=1 < maxf=2: bỏ qua                           | maxf=2 total=4
i=6 x=4: c=2 == maxf: bằng kỷ lục -> total += 2         | maxf=2 total=6
=> kết quả: 6
nums = [1, 2, 2, 3, 1, 4]
i=0 x=1: c=1 > maxf=0: kỷ lục mới -> total = 1          | maxf=1 total=1
i=1 x=2: c=1 == maxf: bằng kỷ lục -> total += 1         | maxf=1 total=2
i=2 x=2: c=2 > maxf=1: kỷ lục mới -> total = 2          | maxf=2 total=2
i=3 x=3: c=1 < maxf=2: bỏ qua                           | maxf=2 total=2
i=4 x=1: c=2 == maxf: bằng kỷ lục -> total += 2         | maxf=2 total=4
i=5 x=4: c=1 < maxf=2: bỏ qua                          

In [16]:
def trace_counter(nums):
    freq = Counter(nums)
    print("freq =", dict(freq))
    mx = max(freq.values())
    print("mx =", mx)
    total = 0
    for v, f in freq.items():
        if f == mx:
            total += f
            print(f"  giá trị {v} (tần suất {f}) đạt mx -> total = {total}")
        else:
            print(f"  giá trị {v} (tần suất {f}) bỏ qua")
    print("=> kết quả:", total)
    return total


assert trace_counter([1, 2, 2, 3, 1, 4]) == 4

freq = {1: 2, 2: 2, 3: 1, 4: 1}
mx = 2
  giá trị 1 (tần suất 2) đạt mx -> total = 2
  giá trị 2 (tần suất 2) đạt mx -> total = 4
  giá trị 3 (tần suất 1) bỏ qua
  giá trị 4 (tần suất 1) bỏ qua
=> kết quả: 4


Bảng tóm tắt (Cách 4, `[1, 2, 2, 3, 1, 4, 4]`):

| `x` | `cnt[x]` | so với `maxf` | `maxf` | `total` |
|---|---|---|---|---|
| 1 | 1 | `1 > 0` kỷ lục | 1 | 1 |
| 2 | 1 | `1 == 1` | 1 | 2 |
| 2 | 2 | `2 > 1` kỷ lục | 2 | 2 |
| 3 | 1 | `1 < 2` | 2 | 2 |
| 1 | 2 | `2 == 2` | 2 | 4 |
| 4 | 1 | `1 < 2` | 2 | 4 |
| 4 | 2 | `2 == 2` | 2 | 6 |

## 9. Code nộp LeetCode

Bản nộp dùng Cách 1 (`Counter`). Đúng class `Solution`, đúng method `maxFrequencyElements(self, nums: List[int]) -> int`.

In [17]:
from typing import List
from collections import Counter

class Solution:
    def maxFrequencyElements(self, nums: List[int]) -> int:
        freq = Counter(nums)                          # giá trị -> số lần xuất hiện
        mx = max(freq.values())                       # tần suất lớn nhất (giá trị, không phải khóa)
        return sum(f for f in freq.values() if f == mx)   # tổng tần suất của các giá trị đạt mx


sol = Solution()
for nums, expected in examples:
    print(nums, "->", sol.maxFrequencyElements(nums))

[1, 2, 2, 3, 1, 4] -> 4
[1, 2, 3, 4, 5] -> 5
[7] -> 1
[5, 5, 5, 2, 2] -> 3


## 10. Vì sao thuật toán đúng

- **Theo định nghĩa.** `freq[v]` là tần suất của giá trị `v`; `mx` là tần suất lớn nhất. "Các phần tử có tần suất lớn nhất" là mọi vị trí của mọi giá trị `v` có `freq[v] == mx`; số vị trí đó là `Σ freq[v]` trên các `v` này, đúng bằng `sum(f for f in freq.values() if f == mx)`.
- **Dạng nhân.** Mỗi giá trị đạt `mx` đóng góp đúng `mx` vị trí, nên đáp án `= mx × k` với `k` là số giá trị đạt `mx`. Vì thế đáp án luôn chia hết cho `mx`.
- **Cách 4 đúng nhờ bất biến.** Sau mỗi bước, `maxf` là tần suất lớn nhất hiện tại và `total` là tổng tần suất hiện tại của các giá trị đạt `maxf`. Tăng `cnt[x]` lên `c`: nếu `c > maxf` thì `x` là giá trị **duy nhất** đạt `c` (mọi giá trị khác `<= maxf < c`), nên `total = c`. Nếu `c == maxf` thì `x` **gia nhập nhóm** với tần suất `c`: `total += c`. Nếu `c < maxf` nhóm không đổi.
- **`mx >= 1`** vì `nums` không rỗng, nên các `cnt[0] = 0` không tham gia ở Cách 3.

## 11. Phân tích Time / Space Complexity

Gọi `n = len(nums)`, `V = 101` là miền giá trị.

- **Cách 1 (`Counter`):** Time `O(n)`, Space `O(n)`.
- **Cách 2 (`dict` thủ công):** Time `O(n)`, Space `O(n)`.
- **Cách 3 (mảng đếm):** Time `O(n + V)`, Space `O(V)`.
- **Cách 4 (một lượt):** Time `O(n)`, Space `O(V)`.
- Phải đọc mọi phần tử nên `O(n)` là tối ưu. Ở đây `n <= 100`.

## 12. Bộ test

In [18]:
sol = Solution()
impls = {
    "counter": max_frequency_elements_counter,
    "dict": max_frequency_elements_dict,
    "array": max_frequency_elements_array,
    "one_pass": max_frequency_elements_one_pass,
    "submit": sol.maxFrequencyElements,
}

for nums, expected in examples:
    for key, fn in impls.items():
        got = fn(list(nums))
        assert got == expected, (key, nums, got, expected)
print("PASS ví dụ của đề (mọi cách)")

PASS ví dụ của đề (mọi cách)


### 12.1. Các trường hợp biên

In [19]:
def check_all(nums, expected, label):
    for key, fn in impls.items():
        got = fn(list(nums))
        assert got == expected, (label, key, got, expected)
    print("PASS", label, "->", expected)


check_all([1], 1, "một phần tử")
check_all([100], 1, "một phần tử lớn nhất")
check_all([3, 3], 2, "hai số giống nhau")
check_all([3, 4], 2, "hai số khác nhau, mỗi số một lần")
check_all([1, 1, 2], 2, "một giá trị thắng rõ rệt")
check_all([1, 1, 2, 2], 4, "hai giá trị cùng tần suất 2")
check_all([1, 2, 2, 3, 3, 3], 3, "tần suất 1, 2, 3: chỉ giá trị 3")
check_all([5, 5, 5, 2, 2, 7], 3, "một giá trị đạt kỷ lục 3, giá trị khác 2 và 1")
check_all([1, 1, 2, 2, 3, 3, 3, 3], 4, "kỷ lục mới ở cuối mảng: chỉ giá trị 3")
check_all([4, 4, 4, 4, 1, 1, 1, 1], 8, "hai giá trị cùng đạt kỷ lục 4")
check_all(list(range(1, 101)), 100, "n = 100 toàn khác nhau")
check_all([100] * 100, 100, "n = 100 toàn một giá trị")
check_all([1] * 50 + [2] * 50, 100, "n = 100, hai giá trị cùng 50 lần")

PASS một phần tử -> 1
PASS một phần tử lớn nhất -> 1
PASS hai số giống nhau -> 2
PASS hai số khác nhau, mỗi số một lần -> 2
PASS một giá trị thắng rõ rệt -> 2
PASS hai giá trị cùng tần suất 2 -> 4
PASS tần suất 1, 2, 3: chỉ giá trị 3 -> 3
PASS một giá trị đạt kỷ lục 3, giá trị khác 2 và 1 -> 3
PASS kỷ lục mới ở cuối mảng: chỉ giá trị 3 -> 4
PASS hai giá trị cùng đạt kỷ lục 4 -> 8
PASS n = 100 toàn khác nhau -> 100
PASS n = 100 toàn một giá trị -> 100
PASS n = 100, hai giá trị cùng 50 lần -> 100


### 12.2. Tính chất và phản ví dụ

In [20]:
random.seed(11)
# Tính chất 1: đáp án = mx * số giá trị đạt mx; chia hết cho mx; 1 <= đáp án <= n
for _ in range(500):
    nums = [random.randint(1, random.choice([3, 10, 100])) for _ in range(random.randint(1, 100))]
    freq = Counter(nums)
    mx = max(freq.values())
    k = sum(1 for f in freq.values() if f == mx)
    ans = sol.maxFrequencyElements(nums)
    assert ans == mx * k and ans % mx == 0 and 1 <= ans <= len(nums)
print("PASS đáp án = mx × số giá trị đạt mx, chia hết cho mx, trong [1, n]")

# Tính chất 2: đáp án = n khi và chỉ khi mọi giá trị có cùng tần suất
for _ in range(500):
    nums = [random.randint(1, 4) for _ in range(random.randint(1, 20))]
    freq = Counter(nums)
    assert (sol.maxFrequencyElements(nums) == len(nums)) == (len(set(freq.values())) == 1)
print("PASS đáp án = n <=> mọi giá trị cùng tần suất")

# Tính chất 3: thứ tự không quan trọng
for _ in range(300):
    nums = [random.randint(1, 10) for _ in range(random.randint(1, 40))]
    shuffled = nums[:]
    random.shuffle(shuffled)
    assert sol.maxFrequencyElements(nums) == sol.maxFrequencyElements(shuffled)
print("PASS thứ tự không quan trọng")

# Tính chất 4: đầu vào không bị sửa
nums = [1, 2, 2, 3, 1, 4]
copy = list(nums)
for key, fn in impls.items():
    fn(nums)
assert nums == copy
print("PASS đầu vào không bị sửa")

# Phản ví dụ 1: trả về tần suất lớn nhất
def sai_tan_suat(nums):
    return max(Counter(nums).values())

assert sai_tan_suat([1, 2, 2, 3, 1, 4]) == 2 != 4
print("PASS phản ví dụ: trả tần suất lớn nhất cho 2 thay vì 4")

# Phản ví dụ 2: trả về số lượng giá trị đạt tần suất lớn nhất
def sai_so_gia_tri(nums):
    freq = Counter(nums)
    mx = max(freq.values())
    return sum(1 for f in freq.values() if f == mx)

assert sai_so_gia_tri([1, 2, 2, 3, 1, 4]) == 2 != 4
print("PASS phản ví dụ: đếm số giá trị cho 2 thay vì 4")

# Phản ví dụ 3: Cách 4 đặt total = 1 khi có kỷ lục mới
def sai_dat_lai(nums):
    cnt, maxf, total = [0] * 101, 0, 0
    for x in nums:
        cnt[x] += 1
        c = cnt[x]
        if c > maxf:
            maxf, total = c, 1                           # sai: phải là c
        elif c == maxf:
            total += c
    return total

assert sai_dat_lai([5, 5, 5]) != 3
print("PASS phản ví dụ: total = 1 cho", sai_dat_lai([5, 5, 5]), "thay vì 3 ở [5, 5, 5]")

PASS đáp án = mx × số giá trị đạt mx, chia hết cho mx, trong [1, n]
PASS đáp án = n <=> mọi giá trị cùng tần suất
PASS thứ tự không quan trọng
PASS đầu vào không bị sửa
PASS phản ví dụ: trả tần suất lớn nhất cho 2 thay vì 4
PASS phản ví dụ: đếm số giá trị cho 2 thay vì 4
PASS phản ví dụ: total = 1 cho 1 thay vì 3 ở [5, 5, 5]


### 12.3. Đối chiếu ngẫu nhiên (có seed)

In [21]:
random.seed(3005)
for trial in range(3000):
    n = random.randint(1, 100)
    hi = random.choice([1, 2, 5, 20, 100])
    nums = [random.randint(1, hi) for _ in range(n)]
    counts = {v: nums.count(v) for v in set(nums)}                          # định nghĩa trực tiếp, bằng list.count
    m = max(counts.values())
    ref = sum(c for c in counts.values() if c == m)
    for key, fn in impls.items():
        assert fn(list(nums)) == ref, (key, nums)
print("PASS 3000 bộ ngẫu nhiên - mọi cách khớp với định nghĩa trực tiếp")

PASS 3000 bộ ngẫu nhiên - mọi cách khớp với định nghĩa trực tiếp


## 13. Các lỗi người mới hay gặp

### Lỗi 1: Trả về tần suất lớn nhất thay vì tổng tần suất

In [22]:
def sai_tra_mx(nums):
    return max(Counter(nums).values())


print("sai  :", sai_tra_mx([1, 2, 2, 3, 1, 4]), "| đúng:", Solution().maxFrequencyElements([1, 2, 2, 3, 1, 4]))

sai  : 2 | đúng: 4


**Vì sao sai:** đề hỏi **tổng tần suất** của **mọi** phần tử đạt tần suất lớn nhất (`2 + 2 = 4`), không phải bản thân tần suất lớn nhất (`2`). Cần cộng thêm bước tổng hợp.

### Lỗi 2: Đếm số giá trị thay vì số lần xuất hiện

In [23]:
def sai_dem_gia_tri(nums):
    freq = Counter(nums)
    mx = max(freq.values())
    return len([v for v in freq if freq[v] == mx])


print("sai  :", sai_dem_gia_tri([1, 2, 2, 3, 1, 4]), "| đúng:", Solution().maxFrequencyElements([1, 2, 2, 3, 1, 4]))

sai  : 2 | đúng: 4


**Vì sao sai:** có `2` **giá trị** (`1` và `2`) đạt tần suất lớn nhất, nhưng mỗi giá trị xuất hiện `2` lần nên có `4` **phần tử** trong mảng. Đáp án cần số **phần tử**, tức `mx × số giá trị`.

### Lỗi 3: `max(freq)` thay vì `max(freq.values())`

In [24]:
def sai_max_khoa(nums):
    freq = Counter(nums)
    mx = max(freq)                                       # khóa lớn nhất, không phải tần suất lớn nhất
    return sum(f for f in freq.values() if f == mx)


print("sai  :", sai_max_khoa([1, 2, 2, 3, 1, 4]), "| đúng:", Solution().maxFrequencyElements([1, 2, 2, 3, 1, 4]))

sai  : 0 | đúng: 4


**Vì sao sai:** `max(freq)` duyệt các **khóa** của dict, cho giá trị lớn nhất xuất hiện trong mảng (`4`), không phải tần suất lớn nhất (`2`). Không có tần suất nào bằng `4`, nên tổng là `0`. Hãy dùng `max(freq.values())`.

### Lỗi 4: Đặt lại `total` sai khi có kỷ lục mới (cách một lượt)

In [25]:
def sai_dat_lai_2(nums):
    cnt, maxf, total = [0] * 101, 0, 0
    for x in nums:
        cnt[x] += 1
        c = cnt[x]
        if c > maxf:
            maxf, total = c, 1                           # chỉ đếm 1 thay vì c
        elif c == maxf:
            total += c
    return total


print("sai  :", sai_dat_lai_2([5, 5, 5, 2, 2]), "| đúng:", Solution().maxFrequencyElements([5, 5, 5, 2, 2]))

sai  : 1 | đúng: 3


**Vì sao sai:** khi `x` lập kỷ lục mới `c`, nhóm đạt kỷ lục chỉ còn `x` với **`c` lần xuất hiện**, nên `total = c` (không phải `1`). Hãy kiểm tra thủ công bằng một mảng ngắn như `[5, 5, 5]`.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- Thấy "tần suất / số lần xuất hiện" → **`Counter`** hoặc mảng đếm.
- Công thức nhớ: **đáp án = `mx × (số giá trị có tần suất mx)`** = `sum(f for f in freq.values() if f == mx)`.
- `max(dict)` là **khóa** lớn nhất; `max(dict.values())` là **giá trị** lớn nhất. Đừng nhầm.
- Phân biệt: "số **giá trị** khác nhau" (đếm khóa) với "số **phần tử**" (cộng tần suất).
- Một lượt duyệt với `maxf` và `total` đang chạy là mẫu hay cho "cực trị + số lượng đạt cực trị".
- Giá trị nhỏ (`<= 100`) → mảng đếm `[0] * 101`, nhanh và đơn giản.

## 15. Biến thể của bài

1. **Tần suất nhỏ nhất**: tổng tần suất của các giá trị có tần suất nhỏ nhất.
2. **Mode(s)**: liệt kê mọi giá trị đạt tần suất lớn nhất.
3. **Top `k`** giá trị xuất hiện nhiều nhất (`Counter.most_common(k)`).
4. **Tần suất lớn thứ hai**.
5. Mảng **đã sắp xếp**: đếm từng đoạn bằng nhau (không cần dict).

Thử biến thể 1, 2 và 5:

In [26]:
def sum_min_frequency(nums):
    freq = Counter(nums)
    mn = min(freq.values())
    return sum(f for f in freq.values() if f == mn)


def modes(nums):
    freq = Counter(nums)
    mx = max(freq.values())
    return sorted(v for v, f in freq.items() if f == mx)


def max_frequency_elements_sorted(nums):
    s = sorted(nums)
    runs = []                                           # độ dài từng đoạn bằng nhau liên tiếp
    i = 0
    while i < len(s):
        j = i
        while j < len(s) and s[j] == s[i]:
            j += 1
        runs.append(j - i)
        i = j
    mx = max(runs)
    return sum(r for r in runs if r == mx)


print("1.", sum_min_frequency([1, 2, 2, 3, 1, 4]))      # 2: các giá trị 3 và 4 (tần suất 1 mỗi cái)
print("2.", modes([1, 2, 2, 3, 1, 4]))                  # [1, 2]
print("5.", max_frequency_elements_sorted([1, 2, 2, 3, 1, 4]))   # 4
assert sum_min_frequency([1, 2, 2, 3, 1, 4]) == 2
assert modes([1, 2, 2, 3, 1, 4]) == [1, 2]
for _ in range(200):
    arr = [random.randint(1, 10) for _ in range(random.randint(1, 40))]
    assert max_frequency_elements_sorted(arr) == Solution().maxFrequencyElements(arr)
print("PASS cách đếm đoạn trên mảng đã sắp xếp khớp lời giải gốc")

1. 2
2. [1, 2]
5. 4
PASS cách đếm đoạn trên mảng đã sắp xếp khớp lời giải gốc


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. `Counter` + `max` + cộng | đếm rồi tổng hợp | `O(n)` | `O(n)` | **cách nộp bài**, ngắn và rõ |
| 2. `dict` thủ công | tự đếm bằng `get` | `O(n)` | `O(n)` | học cơ chế đếm |
| 3. Mảng đếm `[0]*101` | `mx × cnt.count(mx)` | `O(n+V)` | `O(V)` | giá trị nhỏ, nhanh và chắc |
| 4. Một lượt `maxf`/`total` | cập nhật khi đếm | `O(n)` | `O(V)` | muốn một lượt duyệt |

## 17. 5 bài tập vận dụng (dễ → khó)

**Bài 1 (đoán output).** Không chạy code, đoán: `[4, 4, 5].count(4)`, `max({1: 3, 7: 2})`, `max({1: 3, 7: 2}.values())`, `Solution().maxFrequencyElements([2, 2, 3])`, `Solution().maxFrequencyElements([6, 6, 7, 7, 8])`.

**Bài 2 (tính tay).** Với `nums = [3, 1, 3, 2, 1, 3, 2, 2]`: lập bảng tần suất, tìm `mx`, các giá trị đạt `mx` và đáp án.

**Bài 3 (điền TODO).** Hoàn thành `max_freq_todo(nums)` theo Cách 1.

In [27]:
def max_freq_todo(nums):
    # TODO 1: freq = Counter(nums)
    # TODO 2: mx = tần suất lớn nhất (max của freq.values())
    # TODO 3: trả về tổng các tần suất bằng mx
    pass

# KHÔNG gọi hàm TODO ở đây để khi Run All không bị lỗi (các TODO chưa điền nên kết quả chưa đúng).

**Bài 4 (tự viết lại).** Viết `max_freq_one_pass2(nums)` theo **một lượt** (Cách 4) mà không nhìn code mẫu. Kiểm tra với bốn ví dụ trong `examples`.

**Bài 5 (biến thể / mini challenge).** (a) Viết `second_max_frequency_sum(nums)`: tổng tần suất của các giá trị có tần suất **lớn thứ hai** (trả `0` nếu chỉ có một mức tần suất). (b) Viết `top_k_frequent(nums, k)`: `k` giá trị xuất hiện nhiều nhất (hòa thì giá trị nhỏ hơn trước). (c) Viết `max_frequency_elements_stream(stream)` nhận một **iterator** (không biết trước độ dài) và dùng đúng cách một lượt với `dict`.

## 18. Gợi ý 3 mức cho bài tập

- **Mức 1 (nhẹ):** Bài 1: `2`, `7`, `3`, `2` (`[2,2,3]`: `mx = 2`, giá trị `2`, tổng `2`), `4` (`[6,6,7,7,8]`: `mx = 2`, hai giá trị, `2 + 2 = 4`). Bài 3: `freq = Counter(nums)`; `mx = max(freq.values())`; `return sum(f for f in freq.values() if f == mx)`.
- **Mức 2 (vừa):** Bài 2: tần suất `3:3`, `1:2`, `2:3`; `mx = 3`, các giá trị đạt `mx` là `3` và `2`, đáp án `6`. Bài 4: theo mục 6.
- **Mức 3 (sâu):** Bài 5(a): lấy `levels = sorted(set(freq.values()), reverse=True)`; nếu `len(levels) < 2` trả `0`; còn lại tổng các `f == levels[1]`. 5(b): `sorted(freq, key=lambda v: (-freq[v], v))[:k]`. 5(c): duyệt `for x in stream` và dùng `dict` + `maxf`, `total` như Cách 4.

## 19. Đáp án tham khảo cho bài tập

Chạy kiểm tra bằng `assert`.

In [28]:
# Bài 1
assert [4, 4, 5].count(4) == 2 and max({1: 3, 7: 2}) == 7 and max({1: 3, 7: 2}.values()) == 3
assert Solution().maxFrequencyElements([2, 2, 3]) == 2
assert Solution().maxFrequencyElements([6, 6, 7, 7, 8]) == 4

# Bài 2
assert Counter([3, 1, 3, 2, 1, 3, 2, 2]) == Counter({3: 3, 2: 3, 1: 2})
assert Solution().maxFrequencyElements([3, 1, 3, 2, 1, 3, 2, 2]) == 6

# Bài 3
def max_freq_todo_ans(nums):
    freq = Counter(nums)
    mx = max(freq.values())
    return sum(f for f in freq.values() if f == mx)

for nums, expected in examples:
    assert max_freq_todo_ans(nums) == expected

# Bài 4
def max_freq_one_pass2(nums):
    cnt = {}
    maxf = total = 0
    for x in nums:
        cnt[x] = cnt.get(x, 0) + 1
        c = cnt[x]
        if c > maxf:
            maxf, total = c, c
        elif c == maxf:
            total += c
    return total

for nums, expected in examples:
    assert max_freq_one_pass2(nums) == expected

# Bài 5(a)
def second_max_frequency_sum(nums):
    freq = Counter(nums)
    levels = sorted(set(freq.values()), reverse=True)
    if len(levels) < 2:
        return 0
    return sum(f for f in freq.values() if f == levels[1])

assert second_max_frequency_sum([1, 2, 2, 3, 3, 3]) == 2
assert second_max_frequency_sum([1, 1, 2, 2]) == 0
assert second_max_frequency_sum([1, 2, 2, 3, 3, 3, 4, 4]) == 4

# Bài 5(b)
def top_k_frequent(nums, k):
    freq = Counter(nums)
    return sorted(freq, key=lambda v: (-freq[v], v))[:k]

assert top_k_frequent([1, 1, 1, 2, 2, 3], 2) == [1, 2]
assert top_k_frequent([4, 4, 5, 5, 6], 2) == [4, 5]

# Bài 5(c)
def max_frequency_elements_stream(stream):
    cnt = {}
    maxf = total = 0
    for x in stream:
        cnt[x] = cnt.get(x, 0) + 1
        c = cnt[x]
        if c > maxf:
            maxf, total = c, c
        elif c == maxf:
            total += c
    return total

assert max_frequency_elements_stream(iter([1, 2, 2, 3, 1, 4])) == 4
assert max_frequency_elements_stream(x for x in [9, 9, 9]) == 3
print("PASS đáp án phần 19")

PASS đáp án phần 19


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template** (đếm tần suất, cộng các tần suất bằng tần suất lớn nhất)

```python
from typing import List
from collections import Counter

class Solution:
    def maxFrequencyElements(self, nums: List[int]) -> int:
        freq = Counter(nums)
        mx = max(freq.values())
        return sum(f for f in freq.values() if f == mx)
```

**Checklist**

- [ ] Trả về **tổng tần suất** của các giá trị đạt cực đại (không phải `mx`, không phải số giá trị)?
- [ ] Dùng `max(freq.values())` (không phải `max(freq)`)?
- [ ] Nếu một lượt: khi kỷ lục mới đặt `total = c` (không phải `1`), khi bằng kỷ lục `total += c`?
- [ ] Mảng đếm có kích thước `101` (chỉ số `0..100`)?
- [ ] Đã thử: một phần tử, mọi số khác nhau (`n`), mọi số giống nhau (`n`), hai giá trị cùng đạt kỷ lục?
- [ ] Skeleton đúng `class Solution` / `def maxFrequencyElements(self, nums: List[int]) -> int`?